In [1]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types
from tools import get_stock_price

In [2]:
load_dotenv()

api_key = os.environ["GEMINI_API_KEY"]
print("API key loaded:", bool(api_key))

API key loaded: True


In [3]:
def get_stock_price(symbol: str) -> dict:
    """Get the current stock price for a ticker symbol.
    Args:
        symbol: Stock ticker symbol, such as MSFT.
    """

    fake_prices = {
        "MSFT": 512.35,
        "AAPL": 230.10,
        "GOOG": 201.50,
    }

    symbol = symbol.upper()

    if symbol not in fake_prices:
        raise ValueError(f"Unknown stock symbol: {symbol}")

    return {
        "symbol": symbol,
        "price": fake_prices[symbol],
        "currency": "USD",
    }


tool_registry = {
    "get_stock_price": get_stock_price,
}

In [4]:
# Generate Gemini client
client = genai.Client(api_key=api_key)

In [14]:
def execute_function_call(function_call):
    tool_name = function_call.name
    tool = tool_registry.get(tool_name)
    args = dict(function_call.args)

    if tool is None:
        return {
            "status": "error",
            "error": f"Unknown tool: {tool_name}",
        }

    try:
        result = tool(**args)
        return {
            "status": "success",
            "result": result,
        }
    except Exception as e:
        return {
            "status": "error",
            "error": str(e),
        }

In [17]:
def run_agent(user_query, max_steps=10):
    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=user_query)]
        )
    ]

    for step in range(max_steps):
        response = client.models.generate_content(
            model="gemini-flash-lite-latest",
            contents=contents,
            config=types.GenerateContentConfig(
                tools=[get_stock_price],
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                ),
            ),
        )

        model_content = response.candidates[0].content
        contents.append(model_content)

        function_calls = [ 
            part.function_call
            for part in model_content.parts
            if part.function_call is not None
        ]

        # If there is no tool to run, return the text of the last response
        if not function_calls:
            return response.text

        function_response_parts = []
        for function_call in function_calls:
            result = execute_function_call(function_call)

            function_response_parts.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=function_response_parts
            )

        )

    raise RuntimeError(
        f"Agent exceeded maximum number of steps ({max_steps})"
    )

query = "What's Amazon's current stock price?"
final_response = run_agent(query)
print(final_response)

I'm sorry, but I was unable to retrieve the stock price for Amazon (AMZN) at the moment.
